In [1]:
import os
import sys
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
import joblib
import logging

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# Force reload src modules so stale cached versions do not cause ImportError
for _mod in list(sys.modules.keys()):
    if _mod.startswith("src"):
        del sys.modules[_mod]

from src.preprocessing import PromoFeatureEngineer, CompetitionFeatureEngineer, DateFeatureEngineer, DropColumnsTransformer, FillOpenMedian
from src.data_loader import DataLoader


In [2]:
from src.logger import setup_logging
setup_logging(log_dir='../logs')

logger = logging.getLogger('preprocessing')
logger.info('Preprocessing notebook started')

2026-06-13 10:06:23 | INFO     | src.logger | Logging to file: ../logs\run_20260613_100623.log
2026-06-13 10:06:23 | INFO     | preprocessing | Preprocessing notebook started


In [3]:
train_data_path = r"..\data\raw\train.csv"
test_data_path = r"..\data\raw\test.csv"
load_the_data = DataLoader()
df_train = load_the_data.load_data(train_data_path)
df_test = load_the_data.load_data(test_data_path)

2026-06-13 10:06:23 | INFO     | src.data_loader | Loading data from: ..\data\raw\train.csv
2026-06-13 10:06:24 | INFO     | src.data_loader | Loaded 1017209 rows, 9 columns
2026-06-13 10:06:24 | INFO     | src.data_loader | Loading store data from: C:\Users\devan\Desktop\5_DataScience_Project\sales_forecasting\data\raw\store.csv
2026-06-13 10:06:24 | INFO     | src.data_loader | Store data: 1115 rows, 10 columns
2026-06-13 10:06:24 | INFO     | src.data_loader | Merging data with store on 'Store' key (left join)
2026-06-13 10:06:24 | INFO     | src.data_loader | Merge complete — final shape: (1017209, 18)
2026-06-13 10:06:24 | INFO     | src.data_loader | Loading data from: ..\data\raw\test.csv
2026-06-13 10:06:24 | INFO     | src.data_loader | Loaded 41088 rows, 8 columns
2026-06-13 10:06:24 | INFO     | src.data_loader | Loading store data from: C:\Users\devan\Desktop\5_DataScience_Project\sales_forecasting\data\raw\store.csv
2026-06-13 10:06:24 | INFO     | src.data_loader | Store 

In [4]:
print("Training data:-")
print(df_train.isnull().sum())
print("_"*50)
print("Test data:-")
print(df_test.isnull().sum())

Training data:-
Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64
__________________________________________________
Test data:-
Id                               0
Store                            0
DayOfWeek                        0
Date                             0
Open                            11
Promo                            0
StateHoliday                     0
SchoolHoliday  

In [5]:
# Define columns
categorical_cols = [
    'StateHoliday',
    'PromoInterval',
    'StoreType',
    'Assortment'
]

numeric_cols = [
    'DayOfWeek',
    'Open',
    'Promo',
    'SchoolHoliday',
    'CompetitionDistance',
    'Promo2',
    'Promo2SinceDays',
    'CompetitionOpenSinceDays',
    'Year',
    'Month'
]

In [6]:
# Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
            categorical_cols
        ),

        (
            'num',
            'passthrough',
            numeric_cols
        )
    ],
    verbose_feature_names_out=False
).set_output(transform="pandas")

In [7]:
# Full Pipeline
pipeline = Pipeline([
    ('fill_open_column_if_na', FillOpenMedian()),

    ('promo_features', PromoFeatureEngineer()),

    ('competition_features', CompetitionFeatureEngineer()),

    ('date_features', DateFeatureEngineer()),

    ('drop_columns', DropColumnsTransformer()),

    ('preprocessor', preprocessor)

])

In [8]:
# Transform Data
df_train_processed = pipeline.fit_transform(df_train)
df_train_processed['Sales'] = df_train['Sales'].values

2026-06-13 10:06:24 | INFO     | src.preprocessing | [FillOpenMedian] fit — Open column median: 1.0
2026-06-13 10:06:25 | INFO     | src.preprocessing | [FillOpenMedian] transform — no missing Open values found
2026-06-13 10:06:25 | INFO     | src.preprocessing | [PromoFeatureEngineer] engineering promo features — input shape: (1017209, 18)
2026-06-13 10:06:27 | INFO     | src.preprocessing | [CompetitionFeatureEngineer] engineering competition features — input shape: (1017209, 20)
2026-06-13 10:06:28 | WARNING  | src.preprocessing | [CompetitionFeatureEngineer] filling 2642 missing CompetitionDistance values with training mean
2026-06-13 10:06:29 | INFO     | src.preprocessing | [DateFeatureEngineer] extracting Year and Month from Date — input shape: (1017209, 22)
2026-06-13 10:06:30 | INFO     | src.preprocessing | [DropColumnsTransformer] dropping 7 columns: ['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpenDate', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2

In [9]:
saving_path_for_train_df = r"..\data\processed\train.csv"
df_train_processed.to_csv(saving_path_for_train_df, index=False)

In [10]:
df_train_processed.head()

,StateHoliday,PromoInterval,StoreType,Assortment,DayOfWeek,Open,Promo,SchoolHoliday,CompetitionDistance,Promo2,Promo2SinceDays,CompetitionOpenSinceDays,Year,Month,Sales
0,0.0,3.0,2.0,0.0,5,1,1,1,1270.0,0,0.0,2524.0,2015,7,5263
1,0.0,1.0,0.0,0.0,5,1,1,1,570.0,1,1950.0,2829.0,2015,7,6064
2,0.0,1.0,0.0,0.0,5,1,1,1,14130.0,1,1579.0,3164.0,2015,7,8314
3,0.0,3.0,2.0,2.0,5,1,1,1,620.0,0,0.0,2159.0,2015,7,13995
4,0.0,3.0,0.0,0.0,5,1,1,1,29910.0,0,0.0,121.0,2015,7,4822


In [11]:
df_train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [12]:
# Preprocess of df_test data
df_test_processed = pipeline.transform(df_test)

2026-06-13 10:06:39 | WARNING  | src.preprocessing | [FillOpenMedian] filling 11 missing Open values with median 1.0
2026-06-13 10:06:39 | INFO     | src.preprocessing | [PromoFeatureEngineer] engineering promo features — input shape: (41088, 17)
2026-06-13 10:06:39 | INFO     | src.preprocessing | [CompetitionFeatureEngineer] engineering competition features — input shape: (41088, 19)
2026-06-13 10:06:39 | WARNING  | src.preprocessing | [CompetitionFeatureEngineer] filling 96 missing CompetitionDistance values with training mean
2026-06-13 10:06:39 | INFO     | src.preprocessing | [DateFeatureEngineer] extracting Year and Month from Date — input shape: (41088, 21)
2026-06-13 10:06:39 | INFO     | src.preprocessing | [DropColumnsTransformer] dropping 7 columns: ['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpenDate', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2StartDate', 'Date']


In [13]:
saving_path_for_test_df = r"..\data\processed\test.csv"
df_test_processed.to_csv(saving_path_for_test_df, index=False)

In [15]:
# Train pipeline
pipeline.fit(df_train)

# Save the fitted pipeline
joblib.dump(pipeline, "../models/preprocessing_pipeline.pkl")

2026-06-13 10:58:49 | INFO     | src.preprocessing | [FillOpenMedian] fit — Open column median: 1.0
2026-06-13 10:58:50 | INFO     | src.preprocessing | [FillOpenMedian] transform — no missing Open values found
2026-06-13 10:58:50 | INFO     | src.preprocessing | [PromoFeatureEngineer] engineering promo features — input shape: (1017209, 18)
2026-06-13 10:58:51 | INFO     | src.preprocessing | [CompetitionFeatureEngineer] engineering competition features — input shape: (1017209, 20)
2026-06-13 10:58:51 | WARNING  | src.preprocessing | [CompetitionFeatureEngineer] filling 2642 missing CompetitionDistance values with training mean
2026-06-13 10:58:51 | INFO     | src.preprocessing | [DateFeatureEngineer] extracting Year and Month from Date — input shape: (1017209, 22)
2026-06-13 10:58:51 | INFO     | src.preprocessing | [DropColumnsTransformer] dropping 7 columns: ['CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpenDate', 'Promo2SinceYear', 'Promo2SinceWeek', 'Promo2

['../models/preprocessing_pipeline.pkl']

In [16]:
competition_step = pipeline.named_steps['competition_features']

print(hasattr(competition_step, 'competition_distance_mean_'))

True
